# 01 - Data Preparation and Business Decisions

This notebook loads the Online Retail II dataset (UCI Machine Learning
Repository) and establishes the cleaning decisions that all downstream
analyses depend on.

The dataset covers two trading years (Dec. 2009 - Dec. 2011) of a UK-based
online retailer selling giftware, mostly to wholesale customers.

**Purpose of this notebook.** No cleaning is applied at this stage beyond
loading. The goal is first to *observe* the raw data: volume, types, date
range, missingness. Each cleaning decision is then made in its own cell,
justified explicitly, and its impact quantified in both rows removed and
revenue affected. These decisions are recorded in a decision log at the end
of the notebook, because several of them are arbitrary and directly change
the customer segments produced in notebook 02.

**Source file.** The raw `.xlsx` is not versioned in this repository
(~45 MB). It is read once and cached as Parquet, so that re-running the
notebook takes seconds rather than minutes.



# 0. Load Data

In [23]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

parquet_path = PROC / "raw_concat.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    sheets = pd.read_excel(RAW, sheet_name=None)
    frames = []
    for name, frame in sheets.items():
        frame = frame.copy()
        frame["SourceSheet"] = name
        frames.append(frame)
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip() for c in df.columns]

    # The raw file mixes types within object columns:
    #   - 'Invoice' contains integers and cancellation codes ('C489449')
    #   - 'StockCode' contains numeric codes and non-product codes (POST, DOT, M)
    #   - 'Description' contains a few numeric values among free text
    # Mixed types break Parquet writing and would silently corrupt later
    # grouping operations. All object columns are cast to pandas 'string',
    # which preserves missing values as <NA> instead of the literal "nan".
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()

    df.to_parquet(parquet_path, index=False)
print("File loaded")

File loaded


# 1. Data Exploration

## 1.1 Dataset Summary

In [18]:
sheets_list = ", ".join(df['SourceSheet'].unique())
min_date = df['InvoiceDate'].min()
max_date = df['InvoiceDate'].max()

summary = f"""
### 📊 Dataset Summary
* **Dimensions:** `{df.shape[0]:,}` rows × `{df.shape[1]}` columns
* **Source Sheets:** {sheets_list}
* **Date Range:** From `{min_date}` to `{max_date}`
"""
display(Markdown(summary))


### 📊 Dataset Summary
* **Dimensions:** `1,067,371` rows × `9` columns
* **Source Sheets:** Year 2009-2010, Year 2010-2011
* **Date Range:** From `2009-12-01 07:45:00` to `2011-12-09 12:50:00`


## 1.3 Column types

In [25]:
display(Markdown("### 🔠 Column Types"))

dtypes_df = pd.DataFrame(df.dtypes, columns=["Data Type"]).reset_index()
dtypes_df.columns = ["Column", "Type"]

# Display with light formatting (left-aligned text)
display(
    dtypes_df.style
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    .hide(axis="index") # Hide the numeric index on the left
)


### 🔠 Column Types

Column,Type
Invoice,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,string
SourceSheet,string


In [21]:
na = df.isna().sum()
df_missing= pd.DataFrame({
    "n_missing": na, 
    "pct": (na / len(df) * 100).round(2)
}).sort_values(by="n_missing", ascending=False)

display(df_missing)

,n_missing,pct
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
Quantity,0,0.00
StockCode,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
SourceSheet,0,0.00


## First observations

- **1,067,371 transaction lines** across two trading years (Dec. 2009 - Dec. 2011), from a UK-based online retailer selling giftware, mostly to wholesale buyers.
- **`Customer ID` is missing on 22.77% of lines.** These are typically guest checkouts or non-attributed sales. They cannot be used for any customer-level analysis (RFM, retention, churn) but remain valid for product or revenue-level questions. They will be excluded  when the customer table is built in the next step, and the exclusion will be quantified in revenue terms, not just row count.
- **`Description` is missing on 0.41% of lines** negligible, no action needed.
- **Three columns had mixed types in the raw file**: `Invoice` (integers plus `C`-prefixed cancellation codes), `StockCode` (numeric product codes plus non-product codes such as `POST`, `DOT`, `M`), and `Description` (a few numeric values among free text). This is itself informative: the table contains at least two different kinds of events: ordinary sales and cancellations encoded in the same rows rather than as a separate field.
  This is addressed explicitly in the next section.

- **Two columns caused type errors when writing to Parquet**, revealing mixed types not visible from `dtypes` alone (which reports both as generic `object`): `Invoice` (integers plus a `C`-prefixed cancellation code, e.g. `'C489449'`) and `Description` (a few numeric values among free text). Both are cast to a proper string type below. Whether `StockCode` follows the same pattern is checked later, once cancellations are identified.

- This is itself informative: the table contains at least two different kinds
of events — ordinary sales and cancellations — encoded in the same rows
rather than as a separate field.

This includes at least one `C`-prefixed value (`'C489449'`), suggesting a
cancellation convention — confirmed and quantified in the next section.

# Summary Cancellations

In [27]:
# ==========================================
# 🎨 AFFICHAGE ESTHÉTIQUE
# ==========================================
df["IsCancellation"] = df["Invoice"].str.startswith("C")

# 1. Résumé des annulations en Markdown
summary_cancellations = f"""
### ❌ Cancellations Summary
* **Cancellation lines:** `{n_cancel:,}` (`{pct_cancel:.2f}%` of rows)
* **Revenue impact:** `{revenue_cancel:,.2f}` *(vs total signed revenue: `{revenue_total:,.2f}`)*
"""
display(Markdown(summary_cancellations))

# 2. Sanity check avec un tableau stylisé
display(Markdown("### 🔍 Sanity Check: Cancellation Quantities"))
display(Markdown("_Cancellations should carry non-positive quantities._"))

# On convertit le .describe() (qui est une Series) en DataFrame pour le styliser
desc_df = df.loc[df["IsCancellation"], "Quantity"].describe().to_frame(name="Quantity")

# Affichage avec formatage des nombres (séparateurs de milliers et 2 décimales)
display(
    desc_df.style
    .format("{:,.2f}")
    .set_properties(**{'text-align': 'right'})
    .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
)


### ❌ Cancellations Summary
* **Cancellation lines:** `19,494` (`1.83%` of rows)
* **Revenue impact:** `-1,526,667.86` *(vs total signed revenue: `19,287,250.57`)*


### 🔍 Sanity Check: Cancellation Quantities

_Cancellations should carry non-positive quantities._

,Quantity
count,"19,494.00"
mean,-25.19
std,805.10
min,"-80,995.00"
25%,-6.00
50%,-2.00
75%,-1.00
max,1.00


## Decision 1 - Cancellations

Invoices prefixed with `C` are cancellations, not ordinary sales. They carry negative quantities and mirror a prior purchase. 
Two options:
1. Treat them as ordinary negative-quantity rows and let them net out in any revenue or quantity aggregation.
2. Isolate them as a distinct signal (a customer's cancellation rate can itself be informative, e.g. for churn or dissatisfaction), and decide per-analysis whether to net them or exclude them entirely.

This notebook takes option 2: cancellations are flagged in a dedicated boolean column rather than silently netted, so that each downstream notebook can decide explicitly how to treat them instead of inheriting a hidden default.

In [28]:
from IPython.display import display, Markdown

anomaly = df[df["IsCancellation"] & (df["Quantity"] > 0)]
n_anomaly = len(anomaly)

if n_anomaly == 0:
    display(Markdown("### ✅ Sanity Check Passed\n*No anomalous rows found (all cancellations have non-positive quantities).*"))
else:
    display(Markdown(f"### ⚠️ Warning: Anomalies Detected\nFound **`{n_anomaly}`** rows marked as cancellations but with positive `Quantity`."))
    
    display(
        anomaly.head(10).style
        .map(lambda x: 'background-color: #ffe6e6; color: #cc0000; font-weight: bold', subset=['Quantity'])
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    )
    

### ⚠️ Warning: Anomalies Detected
Found **`1`** rows marked as cancellations but with positive `Quantity`.

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,IsCancellation
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.570000,nan,United Kingdom,Year 2009-2010,True


**Anomaly check.** One cancellation row (`C496350`, `StockCode = "M"`: manual entry) carries a positive quantity, contradicting the expected pattern. It has no `Customer ID` attached and will be excluded from any customer-level analysis regardless (see Decision 3 below), so it is left as-is and simply noted here rather than special-cased.

**Conclusion - Decision 1.** Cancellations represent 1.83% of rows but -7.9% of total signed revenue, consistent with a wholesale-heavy customer base where a single cancelled order can be large. They are flagged via `IsCancellation` rather than silently netted, so each downstream notebook (segmentation, retention, prediction) can decide explicitly whether to net them into customer value or treat cancellation behaviour as a signal in
its own right.

## Decision 2 — Non-product stock codes

`StockCode` may mix genuine products with codes representing administrative or non-merchandise entries (postage, manual adjustments, fees). This is checked directly below rather than assumed, since these codes have no meaning for product-level or customer-behaviour analysis.

In [30]:
from IPython.display import display, Markdown

non_product_codes = df.loc[
    df["StockCode"].str.match(r"^[A-Za-z]+$", na=False),
    "StockCode"
].value_counts()

display(Markdown("### 🏷️ Non-Product Codes Overview"))
display(Markdown(f"*Found **`{len(non_product_codes)}`** unique alphabetic stock codes.*"))

npc_df = non_product_codes.reset_index()
npc_df.columns = ["StockCode", "Occurrence Count"]

display(
    npc_df.style
    .format({"Occurrence Count": "{:,}"}) 
    .bar(subset=["Occurrence Count"], color='#5fba7d', vmin=0)
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    .hide(axis="index") # hide numeric index
)

### 🏷️ Non-Product Codes Overview

*Found **`16`** unique alphabetic stock codes.*

StockCode,Occurrence Count
POST,"2,122"
DOT,"1,446"
M,"1,421"
D,177
S,104
ADJUST,67
AMAZONFEE,43
DCGSSGIRL,25
DCGSSBOY,23
PADS,19


`StockCode` mixes genuine products with codes that represent administrative or non-merchandise entries: `POST` (postage), `DOT` (postage/dot com charge), `M` (manual entry), and a few others. These lines have no meaning for product-level or customer-behaviour analysis and are identified explicitly before being excluded from the analytical base.

The alpha-only filter over-catches: `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY` (a childrenswear line) and `PADS` are genuine products, coded with letters only. They are kept. 
`GIFT` (gift card) is also kept as a sellable item, not an accounting entry.

The following are confirmed non-product / administrative codes and are excluded from the analytical base: `POST`, `DOT` (postage), `M`/`m` (manual entries / merged, case-insensitive), `D` (discount), `S` (samples), `ADJUST` (accounting adjustment), `AMAZONFEE` (platform fee), `CRUK` (charity donation).

In [32]:
NON_PRODUCT_CODES = {"POST", "DOT", "M", "D", "S", "ADJUST", "AMAZONFEE", "CRUK"}

df["StockCodeUpper"] = df["StockCode"].str.upper()
df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES)

n_non_product = df["IsNonProduct"].sum()
pct_non_product = n_non_product / len(df) * 100
revenue_non_product = (df.loc[df["IsNonProduct"], "Quantity"] * df.loc[df["IsNonProduct"], "Price"]).sum()

non_product_stats = df.groupby("StockCodeUpper").apply(
    lambda g: pd.Series({"n": len(g), "revenue": (g["Quantity"] * g["Price"]).sum()})
).reindex(list(NON_PRODUCT_CODES)).dropna().reset_index()

non_product_stats.columns = ["Non-Product Code", "Count", "Revenue"]

summary_npc = f"""
### 📦 Non-Product Codes Overview
* **Lines identified:** `{n_non_product:,}` (`{pct_non_product:.2f}%` of rows)
* **Total Revenue Impact:** `{revenue_non_product:,.2f}`
"""
display(Markdown(summary_npc))

display(Markdown("#### Breakdown by Code"))

display(
    non_product_stats.style
    .format({
        "Count": "{:,.0f}", 
        "Revenue": "{:,.2f}"
    })
    .bar(subset=["Count"], color='#a1c9f4', vmin=0) # Barres bleues pour les volumes
    .background_gradient(subset=["Revenue"], cmap='RdYlGn') # Rouge pour négatif, Vert pour positif
    .set_properties(**{'text-align': 'right'})
    .set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    .hide(axis="index")
)

C:\Users\jboul\AppData\Local\Temp\ipykernel_21752\4235625763.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  non_product_stats = df.groupby("StockCodeUpper").apply(



### 📦 Non-Product Codes Overview
* **Lines identified:** `5,401` (`0.51%` of rows)
* **Total Revenue Impact:** `70,795.09`


#### Breakdown by Code

Non-Product Code,Count,Revenue
ADJUST,67,"6,835.24"
D,177,"-13,484.54"
CRUK,16,"-7,933.43"
POST,"2,122","112,341.00"
M,"1,426","-82,781.27"
DOT,"1,446","322,647.47"
AMAZONFEE,43,"-260,763.58"
S,104,"-6,065.80"


**Non-product lines represent 0.51% of rows but 70,795.09 in signed revenue** small in volume, and their net effect happens to be positive, but this masks large offsetting amounts: `DOT` alone contributes +322,647 (postage charges, dotcom operations) while `AMAZONFEE` contributes -260,763 across only 43 lines (average -6,064 per line, large platform-fee debits, not ordinary transactions). `M`, `D`, `S`, and `CRUK` are net negative, as expected for manual adjustments, discounts, samples, and a charity donation.

These lines carry no product or customer-behaviour information and are excluded from the analytical base used in notebooks 02–04. They are kept in the raw table (flagged, not deleted) so the exclusion remains auditable.

## Decision 3 - Missing Customer ID

22.77% of rows have no `Customer ID` (guest checkouts or non-attributed sales). These lines are unusable for any customer-level analysis, RFM, retention, churn, repurchase prediction — since there is no entity to attach them to. They are excluded when building the customer-level table, quantified here in revenue terms rather than row count alone, since excluded rows are not necessarily low-value.

In [33]:
missing_cust = df["Customer ID"].isna()
n_missing = missing_cust.sum()
pct_missing = n_missing / len(df) * 100

revenue_missing = (df.loc[missing_cust, "Quantity"] * df.loc[missing_cust, "Price"]).sum()
revenue_total_all = (df["Quantity"] * df["Price"]).sum()
pct_revenue_missing = revenue_missing / revenue_total_all * 100

summary_missing = f"""
### 👤 Missing Customer IDs Analysis
* **Rows without Customer ID:** `{n_missing:,}` (`{pct_missing:.2f}%` of all rows)
* **Revenue from missing IDs:** `{revenue_missing:,.2f}` (`{pct_revenue_missing:.2f}%` of total signed revenue)
"""

display(Markdown(summary_missing))


### 👤 Missing Customer IDs Analysis
* **Rows without Customer ID:** `243,007` (`22.77%` of all rows)
* **Revenue from missing IDs:** `2,638,958.18` (`13.68%` of total signed revenue)


**Missing-ID rows are lower value on average.** They represent 22.77% of rows but only 13.68% of revenue, roughly 0.60x the average line value of identified transactions. This is consistent with unregistered one-off purchases in a customer base that is otherwise wholesale-heavy, where large buyers are systematically identified. Excluding these rows for customer-level analysis therefore removes a disproportionately low-value, low-signal segment rather than a representative slice of revenue.

## Building the customer-level analytical base

All three decisions are now applied together to produce the transaction table used in notebooks 02–04:

1. Non-product lines (`IsNonProduct`) excluded.
2. Rows without `Customer ID` excluded.
3. Cancellations (`IsCancellation`) are **kept, not netted**, since the choice of whether to net them into customer value or treat cancellation behaviour as a separate signal is analysis-specific and is made again, explicitly, in notebook 02.

In [34]:
clean = df.loc[~df["IsNonProduct"] & df["Customer ID"].notna()].copy()
clean["Customer ID"] = clean["Customer ID"].astype("int64").astype("string")
clean["LineRevenue"] = clean["Quantity"] * clean["Price"]

n_clean = len(clean)
pct_clean = n_clean / len(df) * 100
n_unique_cust = clean['Customer ID'].nunique()
revenue_retained = clean['LineRevenue'].sum()
pct_revenue_retained = revenue_retained / revenue_total_all * 100

out_path = PROC / "clean_transactions.parquet"
clean.to_parquet(out_path, index=False)

summary_clean = f"""
### ✨ Cleaned Dataset Ready
* **Analytical base:** `{n_clean:,}` rows (`{pct_clean:.2f}%` of raw data)
* **Unique customers:** `{n_unique_cust:,}`
* **Revenue retained:** `{revenue_retained:,.2f}` (`{pct_revenue_retained:.2f}%` of total signed revenue)

💾 _Saved successfully to: `{out_path}`_
"""

display(Markdown(summary_clean))


### ✨ Cleaned Dataset Ready
* **Analytical base:** `820,963` rows (`76.91%` of raw data)
* **Unique customers:** `5,882`
* **Revenue retained:** `16,728,575.72` (`86.73%` of total signed revenue)

💾 _Saved successfully to: `..\data\processed\clean_transactions.parquet`_


## Decision log summary

| # | Decision | Rows affected | Revenue affected |
|---|----------|---------------|-------------------|
| 1 | Cancellations flagged, not netted | 19,494 (1.83%) | -1,526,667.86 (-7.9%) |
| 2 | Non-product codes excluded | 5,401 (0.51%) | +70,795.09 |
| 3 | Missing Customer ID excluded | 243,007 (22.77%) | +2,638,958.18 (13.68%) |

The resulting analytical base (`clean_transactions.parquet`) keeps cancellations as a flagged signal and drops only non-product lines and unidentified customers, the two categories with no customer-level meaning.
This table is the single source for notebooks 02-04.

## Summary

The analytical base retains 76.91% of raw rows but 86.73% of revenue, across 5,882 uniquely identified customers spanning Dec. 2009 – Dec. 2011. This confirms the earlier observation: excluded rows (non-product entries, unidentified customers) are disproportionately low-value. This table (`clean_transactions.parquet`) is the single input to all following notebooks.